Identify any additional required reactions associated with the required metabolites. This mainly is checking whether any producing/consuming reactions do not exist, and if they do exist, ensuring they are not blocked:

In [3]:
import urllib
import requests
import json
import os

import pandas as pd
import cobra

from human_me import io
from human_me.data.file_paths import input_local_path, build_files_url, build_local_path
from human_me.utils.functions import flatten_list
from human_me.preprocess.correct_inputs import correct_model

In [4]:
n_cores = 20

Load the data:

In [42]:
with urllib.request.urlopen(build_files_url + "required_metabolic_model_metabolites.json") as url:
    required_metabolites = json.loads(url.read().decode())
required_metabolites = flatten_list([v for v in required_metabolites.values()])

fn = 'https://raw.githubusercontent.com/hmbaghdassarian/human_me_data/master/required_metabolites_consumed_in_Ematrix.txt'
consumed_metabolites_all = requests.get(fn).text.splitlines()

fn = 'https://raw.githubusercontent.com/hmbaghdassarian/human_me_data/master/required_metabolites_produced_in_Ematrix.txt'
produced_metabolites_all = requests.get(fn).text.splitlines()

recon2 = io.load_metabolic_model(os.path.join(input_local_path, 'recon2_2.xml'))

psim_gold = pd.read_hdf(build_local_path + 'psim_gold.h5')
psim_gold = psim_gold[psim_gold.Status != 0]  # drop genes that won't work with model

Our main concern is with metabolites consumed by the E-matrix but not produced by it, and vice-versa. These will require associated metabolic module reactions to carry flux:

In [49]:
produced_metabolites= list(set(produced_metabolites_all).difference(consumed_metabolites_all))
consumed_metabolites = list(set(consumed_metabolites_all).difference(produced_metabolites_all))

print('{} of {} metabolites produced by the expression module are not consumed by the expression module'.format(len(produced_metabolites), len(produced_metabolites_all)))
print('{} of {} metabolites produced by the expression module are not consumed by the expression module'.format(len(consumed_metabolites), len(consumed_metabolites_all)))


132 of 155 metabolites produced by the expression module are not consumed by the expression module
28 of 51 metabolites produced by the expression module are not consumed by the expression module


In [10]:
# correct model is run prior to adding the gapfilling for blocked reactions (in this notebook)
# commit ID: 857b22367d2e4a8529b1367cf7e2c68d05e3ae7b
cm_1, cm_2, _ = correct_model(recon2.copy(), 
                           correct_biomass = True
                          )

/data2/hratch/Software/me_analyses_1_env/lib/python3.9/site-packages/human_me/preprocess/correct_inputs.py:121 UserWarning: ACCOAC contains redundant complexes according to GPR, editing GPR
/data2/hratch/Software/me_analyses_1_env/lib/python3.9/site-packages/human_me/preprocess/correct_inputs.py:121 UserWarning: OIVD1m contains redundant complexes according to GPR, editing GPR
/data2/hratch/Software/me_analyses_1_env/lib/python3.9/site-packages/human_me/preprocess/correct_inputs.py:121 UserWarning: OIVD2m contains redundant complexes according to GPR, editing GPR
/data2/hratch/Software/me_analyses_1_env/lib/python3.9/site-packages/human_me/preprocess/correct_inputs.py:121 UserWarning: OIVD3m contains redundant complexes according to GPR, editing GPR
/data2/hratch/Software/me_analyses_1_env/lib/python3.9/site-packages/human_me/preprocess/correct_inputs.py:121 UserWarning: PFK contains redundant complexes according to GPR, editing GPR
/data2/hratch/Software/me_analyses_1_env/lib/python3.

Check for the recon2.2 HGNC:HGNC error
Remove genes not participating in reactions


/data2/hratch/Software/me_analyses_1_env/lib/python3.9/site-packages/human_me/preprocess/correct_inputs.py:167 UserWarning: Your metabolic model contains genes with HGNC:HGNC:####, changing to HGNC:####


In [11]:
for reaction in cm_2.reactions:
    if reaction.upper_bound > 1000:
        reaction.upper_bound = 1000
    if reaction.lower_bound < -1000:
        reaction.lower_bound = -1000

In [12]:
corrected_reactions_1 = list({r.id for r in cm_1.reactions}.difference([r.id for r in recon2.reactions])) # just naming conventions
corrected_reactions = {r.id for r in cm_2.reactions}.difference([r.id for r in recon2.reactions] + corrected_reactions_1)

Get all blocked reactions in the full model:

In [50]:
blocked_reactions = cobra.flux_analysis.find_blocked_reactions(model = cm_2, 
                                                              open_exchanges = True, 
                                                              processes=n_cores)

# Consumed Metabolites

For the consumed metabolites, let's see what the producing reactions are:

In [100]:
producing_reactions = {}
for m_id in consumed_metabolites:
    m_obj = cm_2.metabolites.get_by_id(m_id)
    producing_reactions[m_id] = []
    
    for reaction in m_obj.reactions:
        if m_obj in reaction.products or reaction.reversibility:
            producing_reactions[m_id] += [reaction.id]

no_metabolic_producers = [m_id for m_id, v in producing_reactions.items() if len(v) == 0]
msg = '{} of {} metabolites consumed by the expression module AND not produced by the expression module '
msg += 'have no producing reactions in the metabolic module'
print(msg.format(len(no_metabolic_producers), len(consumed_metabolites)))

0 of 28 metabolites consumed by the expression module AND not produced by the expression module have no producing reactions in the metabolic module


In [101]:
blocked_producers = {}
for m_id, pr in producing_reactions.items():
    if m_id not in no_metabolic_producers:
        if len(set(pr).difference(blocked_reactions))==0:
            blocked_producers[m_id] = pr

msg = '{} of {} metabolites consumed by the expression module AND not produced by the expression module '
msg += 'have blocked producing reactions in the metabolic module'
print(msg.format(len(blocked_producers), len(consumed_metabolites)))

blocked_producers

2 of 28 metabolites consumed by the expression module AND not produced by the expression module have blocked producing reactions in the metabolic module


{'atp_l': ['r0859'], 'amet_n': ['AMETtn']}

It looks like lysosomal ATP and nuclear adenosyl methionine cannot carry flux in the full model, but their products are required consumed metabolites in the ME model. Let's solve this.

For lysosomal ATP, the transport reaction has as a substrate adp_l, however, there is no producing reaction for adp_l. The additional reactions associated with adp_l in Recon3D only consume it. 

In [16]:
backup = cm_2.copy()

In [17]:
print(cm_2.reactions.get_by_id('r0859').reaction)
print(cm_2.metabolites.get_by_id('adp_l').reactions)

adp_l + atp_c --> adp_c + atp_l
frozenset({<Reaction r0859 at 0x7f3069436520>})


From a literature search ([1](https://doi.org/10.1073/pnas.080014110), [2](https://doi.org/10.3390/cells11050887)), SLC17A9 (HGNC:16192) acts as a transporter of ATP to the lysosome. Thus, we add this reaction. For the reaction not to be blocked, it also needs to be able to leave the lysosome. The mechanism of SLC17A9 and associated transporters is eventual secretion to the ECM via vesicles, so we include this reaction as well.

In [ ]:
if 'HGNC:16192' not in psim_gold.HGNC_ID.tolist():
    raise ValueError('Cannot add this gpr')

In [18]:
m_model = cm_2.copy()
reaction = cobra.Reaction(id='ATPtl', 
                          lower_bound=0, 
                          upper_bound = 1000, 
                         name = 'ATP transporter, lysosomal')
reaction.gene_reaction_rule = 'HGNC:16192'
m_model.add_reactions([reaction.copy()])
reaction = m_model.reactions.get_by_id('ATPtl')
reaction.add_metabolites({'atp_c': -1, 'atp_l': 1})

reaction = cobra.Reaction(id='ATPte', 
                          lower_bound=0, 
                          upper_bound = 1000, 
                         name = 'ATP transporter, extracellular')
reaction.gene_reaction_rule = 'HGNC:16192'
m_model.add_reactions([reaction.copy()])
reaction = m_model.reactions.get_by_id('ATPtl')
reaction.add_metabolites({'atp_l': -1, 'atp_e': 1})

Next, let's recalculate the blocked reactions:

In [32]:
# m_model.add_boundary(metabolite = m_model.metabolites.get_by_id('atp_l'), 
#                          type='demand')

In [55]:
blocked_reactions_2 = cobra.flux_analysis.find_blocked_reactions(model = m_model, 
                                                              open_exchanges = True, 
                                                              processes=n_cores)

In [56]:
set(blocked_reactions).difference(blocked_reactions_2)

set()

In [57]:
set(blocked_reactions_2).difference(blocked_reactions)

set()

In [58]:
'ATPtl' in blocked_reactions_2

False

In [59]:
'ATPte' in blocked_reactions_2

False

Next, let's consider amet_n. It can only be produced by amet_c. However, reactions producing amet_c can carry flux, so this is not the problem. 

In [55]:
# print(cm_2.reactions.get_by_id('AMETtn').reaction)

# producing_amet_c = []
# for r in cm_2.metabolites.get_by_id('amet_c').reactions:
#     if r.metabolites[cm_2.metabolites.get_by_id('amet_c')] > 0:
#         producing_amet_c.append(r.id)
# if len(set(producing_amet_c).difference(blocked_reactions)) > 1: 
#     print('amet_c can be produced by the model')

# Produced Metabolites

In [103]:
consuming_reactions = {}
for m_id in produced_metabolites:
    m_obj = cm_2.metabolites.get_by_id(m_id)
    consuming_reactions[m_id] = []
    
    for reaction in m_obj.reactions:
        if m_obj in reaction.reactants or reaction.reversibility:
            consuming_reactions[m_id] += [reaction.id]

no_metabolic_consumers = [m_id for m_id, v in consuming_reactions.items() if len(v) == 0]
msg = '{} of {} metabolites produced by the expression module AND not consumed by the expression module '
msg += 'have no consuming reactions in the metabolic module'
print(msg.format(len(no_metabolic_consumers), len(produced_metabolites)))

no_metabolic_consumers

1 of 132 metabolites produced by the expression module AND not consumed by the expression module have no consuming reactions in the metabolic module


['thr_L_m']

In [109]:
blocked_consumers = {}
for m_id, pr in consuming_reactions.items():
    if m_id not in no_metabolic_consumers:
        if len(set(pr).difference(blocked_reactions))==0:
            blocked_consumers[m_id] = pr

msg = '{} of {} metabolites produced by the expression module AND not consumed by the expression module '
msg += 'have blocked consuming reactions in the metabolic module'
print(msg.format(len(blocked_consumers), len(produced_metabolites)))

blocked_consumers

79 of 132 metabolites produced by the expression module AND not consumed by the expression module have blocked consuming reactions in the metabolic module


{'arg_L_x': ['arg_Ltx'],
 'pro_L_r': ['PROAKGOX1r'],
 'ser_L_m': ['GHMT2rm'],
 'gln_L_n': ['gln_Ltn'],
 'cys_L_n': ['cys_Ltn'],
 'thr_L_r': ['thr_Ltr'],
 'met_L_n': ['met_Ltn'],
 'phe_L_r': ['phe_Ltr'],
 'asn_L_l': ['r1061'],
 'arg_L_n': ['arg_Ltn'],
 'cys_L_r': ['cys_Ltr'],
 'thr_L_x': ['thr_Ltx'],
 'val_L_l': ['r1140'],
 'met_L_l': ['r1073'],
 'trp_L_l': ['r1077'],
 'val_L_r': ['val_Ltr'],
 'met_L_m': ['RE2030M'],
 'ser_L_r': ['RE3301R'],
 'ahcys_n': ['AHCYStn'],
 'ser_L_n': ['ser_Ltn'],
 'leu_L_n': ['leu_Ltn'],
 'thr_L_n': ['thr_Ltn'],
 'pro_L_x': ['pro_Ltx'],
 'leu_L_x': ['leu_Ltx'],
 'val_L_n': ['val_Ltn'],
 'phe_L_n': ['phe_Ltn'],
 'thr_L_l': ['r0925'],
 'leu_L_l': ['r1068'],
 'gly_l': ['GLYt2rL'],
 'tyr_L_l': ['r1080'],
 'asn_L_r': ['asn_Ltr'],
 'ile_L_l': ['r0825'],
 'his_L_l': ['HIShPTtc', 'r1067'],
 'asn_L_x': ['asn_Ltx'],
 'gln_L_r': ['gln_Ltr'],
 'gly_n': ['glytn'],
 'his_L_r': ['his_Ltr'],
 'ile_L_n': ['ile_Ltn'],
 'asp_L_r': ['asp_Ltr'],
 'glu_L_x': ['glu_Ltx'],
 'phe_L_l